In [1]:
from dotenv import load_dotenv,find_dotenv
from langchain.tools import tool
from langchain.messages import AIMessage, HumanMessage, SystemMessage
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser, BaseOutputParser
from langchain_core.runnables import RunnableParallel, RunnableLambda
from langchain_core.exceptions import OutputParserException
import os
import json
import pandas as pd
from pathlib import Path
import numpy as np

load_dotenv(find_dotenv())

model = ChatGoogleGenerativeAI(
    model="gemini-3-flash-preview",
    temperature=1.0,
    max_tokens=None,
    timeout=None,
    max_retries=2,
)

ImportError: module ''langchain_core._api'.'deprecation'' not found (No module named 'pydantic_core._pydantic_core')

In [3]:
template = """
You are an expert data scientist with an expertise in building deep learning models. 
Explain the concept of {concept} in a couple of lines
"""

prompt = PromptTemplate(
    input_variables=["concept"],
    template=template,
)

response = model.invoke(prompt.format(concept="molecular biology"))
print(response)

content=[{'type': 'text', 'text': 'From a data perspective, **molecular biology** is the study of the "source code" of life; it investigates how information flows between DNA, RNA, and proteins to drive cellular function. It focuses on the chemical mechanisms and regulatory networks that govern how these macromolecules interact to encode and execute biological programs.', 'extras': {'signature': 'EpcKCpQKAb4+9vsfzq2PPEgm8ziV7SjkckXzOWKx2WwCMNUHVnZta5nqVa8tbL98Y0NHRo+qMqd4BpKbs6/SaQt4HR84Sjkv258Jg9ZwY38LLokXFfO6WG00Hmv3ilHow+TWcqSUSxFR1F9yteIniZIVdIs9d0qfqlt6yFKXE73XyL+P5vj0YR2plkW+9C+Ht/bqkTk+Ge9i+kx+rxCODrmZ4kbnHpPBgNsPRMT0bxVAjJp16LvJrDoa+0RayUMbJa2+GrwbNns35w6lp1riLffvyFUZmebE1kJjMH42xorioWJ3hTcPdkDuNSGinTDNVeKjIfUxAZPqt+4yiOaSEooQ0f9HvpZQx5CBjp/nkpHcnNS8pbE9pkDuVGfNhmYBmuGkE4Zy4N7f/kSJwLb/yB4/H+uyOAI5AMoRSKrrLnXUntMh5FeUKpyolWvQ0SdjYNzc1O6U7tpaKWxeePMCEBkgw4UbAnREpnLL0DrH2Y/t+B95E3x7CId6qGCgVV5uShV+9ZFsg09B7YspyK5r/+uOWKSnufpVNkBU1Pb+M3KXMcF/MFlxVsTZphMZlQMu0tiYFcDi8bJzmAJ6xadAlp5v

In [14]:
# Define prompt template
prompt_template = ChatPromptTemplate.from_messages(
    [
        ("system", "You are an expert product reviewer."),
        ("human", "List the main features of the product {product_name}."),
    ]
)


# Define pros analysis step
def analyze_pros(features):
    pros_template = ChatPromptTemplate.from_messages(
        [
            ("system", "You are an expert product reviewer."),
            (
                "human",
                "Given these features: {features}, list the pros of these features.",
            ),
        ]
    )
    return pros_template.format_prompt(features=features)


# Define cons analysis step
def analyze_cons(features):
    cons_template = ChatPromptTemplate.from_messages(
        [
            ("system", "You are an expert product reviewer."),
            (
                "human",
                "Given these features: {features}, list the cons of these features.",
            ),
        ]
    )
    return cons_template.format_prompt(features=features)


# Combine pros and cons into a final review
def combine_pros_cons(pros, cons):
    return f"Pros:\n{pros}\n\nCons:\n{cons}"


# Simplify branches with LCEL
pros_branch_chain = (
    RunnableLambda(lambda x: analyze_pros(x)) | model | StrOutputParser()
)

cons_branch_chain = (
    RunnableLambda(lambda x: analyze_cons(x)) | model | StrOutputParser()
)

# Create the combined chain using LangChain Expression Language (LCEL)
chain = (
    prompt_template
    | model
    | StrOutputParser()
    | RunnableParallel(branches={"pros": pros_branch_chain, "cons": cons_branch_chain})
    | RunnableLambda(lambda x: combine_pros_cons(x["branches"]["pros"], x["branches"]["cons"]))
)

# Run the chain
result = chain.invoke({"product_name": "MacBook Pro"})

# Output
print(result)

Pros:
Based on the features provided, here are the **pros** of the current MacBook Pro (M3 family) from an expert product reviewer’s perspective:

### 1. Unmatched Performance Efficiency
*   **Unified Memory Architecture (UMA):** By allowing the CPU and GPU to share a single pool of memory, the system eliminates data duplication, resulting in significantly faster performance and lower latency.
*   **Scalable Power:** Whether you are a student (M3) or a high-end 3D animator (M3 Max), the chip lineup offers a specific tier of power tailored to your workload.
*   **Next-Gen Graphics:** Hardware-accelerated ray tracing brings the MacBook Pro up to speed with high-end gaming PCs, making it a viable machine for game developers and 3D artists.

### 2. The Gold Standard Display
*   **Desktop-Quality Visuals:** The Mini-LED technology delivers "true blacks" and a contrast ratio that rivals high-end OLED monitors, essential for color-grading and photo editing.
*   **Ultra-Smooth Interaction:** P

In [16]:
prompt_template = ChatPromptTemplate.from_messages(
    [
        ("system", "Tu es un assistant à la conception de règles pour un jeu de rôle. Ta fonction est de concevoir la composition d'une faculté parapsychique en fonction de sa description. N'invente aucune mécanique de jeu et aucun détail spécifique sur le contexte du jeu. Ne donne pas d'example. Reste concis, et ne donne pas d'information supplémentaire. Voici la structure des faculté télépathiques : {STRUCTURE_TELEPATHY}"),
        ("human", "Voici le fonctionnement de la faculté : {ability_description}."),
    ]
)

chain = (
    prompt_template
    | model
    | StrOutputParser()
)

result = chain.invoke({"STRUCTURE_TELEPATHY":"Détection - Accès - Action", "ability_description":"écouter le dialogue interne d'une personne"})

print(result)

**Détection** : Présence mentale
**Accès** : Connexion télépathique
**Action** : Écoute du dialogue interne


In [13]:
def make_prompts(path: Path, ability_description: str) -> list[dict[str,str]]:
    with open(path, "r") as f:
        details = json.load(f)
    details = details['telepathy']
    result = []

    list_names = {}
    for index, parapsy_list in enumerate(details['lists']):
        list_names[f'list_name_{index+1}'] = parapsy_list['list_name']
        list_names[f'list_description_{index+1}'] = parapsy_list['list_description']

    for parapsy_list in details['lists']:
        for parapsy_sublist in parapsy_list['sublists']:
            
            prompt_terms = {
                "ability_description":ability_description,
                "mode_adjective":details['adjective'],
                "mode_noun":details['noun'],
                "mode_general_description":details['general_description'],
                "list_name_1":list_names['list_name_1'],
                "list_description_1":list_names['list_description_1'],
                "list_name_2":list_names['list_name_2'],
                "list_description_2":list_names['list_description_2'],
                "list_name_3":list_names['list_name_3'],
                "list_description_3":list_names['list_description_3'],
                "sublist_name":parapsy_sublist['sublist_name'],
                "list_name":parapsy_list['list_name'],
                "sublist_description":parapsy_sublist['sublist_description']
            }

            for item_index, item in enumerate(parapsy_sublist['items']):
                prompt_terms[f'item_{item_index+1}'] = item['item_name']
                prompt_terms[f'item_description_{item_index+1}'] = item['item_description']
            
        result.append(prompt_terms)
    
    return result


In [26]:
prompt_template = ChatPromptTemplate.from_messages(
    [
        ("system", """
Tu es un assistant à la conception de règles pour un jeu de rôle. Ta fonction est d'estimer une grandeur liée à un certain aspect d'un faculté {mode_adjective}. La {mode_noun} peut être décrite de la manière suivante : {mode_general_description}. Chaque faculté se décompose en trois listes, dont voici les détails : 
- {list_name_1} : {list_description_1}
- {list_name_2} : {list_description_2}
- {list_name_3} : {list_description_3}
Tu dois évaluer la valeur associée à la sous-liste nommée {sublist_name}, contenue dans {list_name}, dont la définition est la suivante : {sublist_description}.
Tu as le choix entre ces cinq valeurs :
- 1
{item_1} : {item_description_1}
- 2
{item_2} : {item_description_2}
- 3
{item_3} : {item_description_3}
- 4
{item_4} : {item_description_4}
- 5
{item_5} : {item_description_5}
Réponds en détaillant ton raisonnement.
        """),
        ("human", "Voici le fonctionnement de la faculté : {ability_description}."),
    ]
)

chain = (
    prompt_template
    | model
    | StrOutputParser()
)

prompt = make_prompts(
    path=Path("/Users/frengineer/Documents/Other/ability_maker/parapsy_lists2.json"),
    ability_description="Le personnage est capable de lire dans les pensées de n'importe quel individu qu'il regarde dans les yeux"
)[0]

print(prompt)

print(prompt_template.invoke(prompt))

#result = chain.invoke(prompt)

#print(result)

{'ability_description': "Le personnage est capable de lire dans les pensées de n'importe quel individu qu'il regarde dan les yeux", 'mode_adjective': 'télépathique', 'mode_noun': 'télépathie', 'mode_general_description': "Discipline permettant de détecter les esprits, d'accéder à certains de leurs sous-systèmes et d'effectuer des actions sur ces derniers. De la même manière qu'un individu peut agir sur ses propres pensées, son corps, ses souvenirs, etc, le télépathe a un contrôle plus profond et qui peut s'étendre à d'autres esprits que le sien", 'list_name_1': 'Détection', 'list_description_1': 'Détection de la conscience de créatures biologiques. La distance, à la fois littérale que figurative, le nombre des individus détectés, ainsi que la nature des esprits viennent influer sur la capacité du télépathe à les détecter', 'list_name_2': 'Accès', 'list_description_2': "Accès aux différents niveaux de l'arborescence des systèmes nerveux. Certains composants son plus proches de la surfac

In [ ]:
class BooleanOutputParser(BaseOutputParser[bool]):
    true_val: str = "YES"
    false_val: str = "NO"

    def parse(self, text: str) -> bool:
        cleaned_text = text.strip().upper()
        if cleaned_text not in (
            self.true_val.upper(),
            self.false_val.upper(),
        ):
            raise OutputParserException(
                f"BooleanOutputParser expected output value to either be "
                f"{self.true_val} or {self.false_val} (case-insensitive). "
                f"Received {cleaned_text}."
            )
        return cleaned_text == self.true_val.upper()

    @property
    def _type(self) -> str:
        return "boolean_output_parser"

In [1]:
import pandas as pd

with open("/Users/dendarii/Documents/Code/ability_maker/parapsy_lists.json", mode="r") as f:
    data = json.load(f)

columns = ["item_index","name","description","description_alt"]
data_lists = []
for schema in data['schemas']:
    data_lists.append({
        "item_index":"",
        "name":schema["noun"],
        "description":schema["general_description"],
        "description_alt":""
        })
    for parapsy_list in schema['lists']:
        data_lists.append({
            "item_index":"",
            "name":parapsy_list["list_name"],
            "description":parapsy_list["list_description"],
            "description_alt":""
            })
        for parapsy_sublist in parapsy_list['sublists']:
            data_lists.append({
                "item_index":"",
                "name":parapsy_sublist["sublist_name"],
                "description":parapsy_sublist["sublist_description"],
                "description_alt":""
                })
            for items in parapsy_sublist['items']:
                data_lists.append({
                    "item_index":items["item_value"],
                    "name":items["item_description"],
                    "description":items["item_description_alt"],
                    "description_alt":""
                    })
    print(schema)


dataframe = pd.DataFrame(data_lists, columns=columns)
dataframe

ImportError: Unable to import required dependency numpy. Please see the traceback for details.